In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras import layers, models
import json
import os

# 🔹 CONFIG
DATASET_PATH = "PlantVillage"   # 👉 CHANGE THIS to your dataset folder

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 50   # increase for better accuracy

# 🔹 Check dataset
if not os.path.exists(DATASET_PATH):
    raise ValueError("Dataset path not found!")

# 🔹 Data Generator (with augmentation)
# ⚠️ Changed: Using MobileNetV2's specific preprocessing instead of basic 1/255 rescaling
datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

train_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

print("Classes:", train_data.class_indices)

# 🔥 MODEL (MobileNetV2 - Lighter and Faster)
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base model
base_model.trainable = False

# Add custom layers
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
output = layers.Dense(train_data.num_classes, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

# 🔹 Compile
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 🔹 Train
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS
)



Found 16515 images belonging to 15 classes.
Found 4122 images belonging to 15 classes.
Classes: {'Pepper__bell___Bacterial_spot': 0, 'Pepper__bell___healthy': 1, 'Potato___Early_blight': 2, 'Potato___Late_blight': 3, 'Potato___healthy': 4, 'Tomato_Bacterial_spot': 5, 'Tomato_Early_blight': 6, 'Tomato_Late_blight': 7, 'Tomato_Leaf_Mold': 8, 'Tomato_Septoria_leaf_spot': 9, 'Tomato_Spider_mites_Two_spotted_spider_mite': 10, 'Tomato__Target_Spot': 11, 'Tomato__Tomato_YellowLeaf__Curl_Virus': 12, 'Tomato__Tomato_mosaic_virus': 13, 'Tomato_healthy': 14}
Epoch 1/2
517/517 ━━━━━━━━━━━━━━━━━━━━ 1676s 3s/step - accuracy: 0.7776 - loss: 0.6980 - val_accuracy: 0.8819 - val_loss: 0.3486
Epoch 2/2
517/517 ━━━━━━━━━━━━━━━━━━━━ 815s 2s/step - accuracy: 0.8624 - loss: 0.4160 - val_accuracy: 0.9054 - val_loss: 0.2655


In [3]:
# 🔥 Save Model
model.save("plant_disease_model_mobilenet.h5")

# 🔹 Save class labels
with open("class_labels.json", "w") as f:
    json.dump(train_data.class_indices, f)

print("\n✅ TRAINING COMPLETE")
print("📁 Model saved as: plant_disease_model_mobilenet.h5")
print("📁 Labels saved as: class_labels.json")


✅ TRAINING COMPLETE
📁 Model saved as: plant_disease_model_mobilenet.h5
📁 Labels saved as: class_labels.json
